# Lezione 12 — Innovazione responsabile e intelligenza artificiale (IA): etica, bias e interpretabilità

Quaderno della lezione: lo proietti ed esegui dal vivo. Alterna una breve spiegazione teorica con celle di codice funzionanti.

## Obiettivi di apprendimento

- Riconosci come un bias algoritmico si insinua in un sistema di IA senza che nessuno lo cerchi di proposito
- Costruisci un dataset sintetico con due gruppi demografici in cui uno porta uno svantaggio storico
- Addestri un classificatore semplice con scikit-learn su quel dataset
- Calcoli una metrica di equità standard, la differenza di pari opportunità, e la interpreti
- Visualizzi il divario tra gruppi con un grafico a barre

## Quando un modello impara un bias

Nessun team progetta un sistema per discriminare. Il bias entra dalla porta di servizio: dati poco rappresentativi, etichette che ereditano pregiudizi storici e metriche di successo che non includono l'equità come obiettivo.

Casi reali che conosci già dal manuale:

- Il sistema COMPAS (Correctional Offender Management Profiling for Alternative Sanctions), usato nei tribunali degli Stati Uniti, segnalava le persone afroamericane come ad alto rischio con un tasso di falsi positivi molto superiore a quello della popolazione bianca
- Nel 2015, il riconoscimento delle immagini di Google Photos ha etichettato come "gorilla" fotografie di persone nere
- Strumenti di ricerca di lavoro hanno mostrato annunci di posizioni ben retribuite agli uomini con una frequenza molto maggiore che alle donne

In tutti e tre i casi, il processo tecnico funzionava "bene" secondo la sua metrica di addestramento. Il problema stava nei dati e in ciò che si era deciso di misurare. In questo quaderno riproduci questo schema su piccola scala: generi un dataset con un bias di rappresentazione deliberato, addestri un modello su di esso e misuri quanto l'effetto si noti.

## Genera un dataset sintetico con due gruppi

Crei un problema di classificazione binaria con due gruppi demografici fittizi, A e B. Il processo causale che genera il risultato reale è identico nei due gruppi, ma i dati di origine del gruppo B riflettono valori sistematicamente più bassi in una variabile chiave e più rumore di misurazione nell'etichetta. Così simuli un bias di rappresentazione all'origine dei dati, non un bias introdotto dal modello.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_por_grupo = 1000

# --- Grupo A: grupo mayoritario, bien representado en el histórico ---
edad_a = rng.normal(40, 10, n_por_grupo)
ingresos_a = rng.normal(50, 15, n_por_grupo)
antiguedad_a = rng.normal(8, 4, n_por_grupo)
score_a = 0.04 * ingresos_a + 0.3 * antiguedad_a - 5
etiqueta_a = (score_a + rng.normal(0, 3, n_por_grupo) > 0).astype(int)

# --- Grupo B: grupo minoritario, peor representado en el histórico ---
# El proceso causal real es el mismo, pero los datos disponibles reflejan
# ingresos declarados sistemáticamente más bajos y una etiqueta con más
# ruido de medición: así se simula el sesgo de representación en el origen
# de los datos, no un sesgo del modelo en sí mismo.
edad_b = rng.normal(38, 10, n_por_grupo)
ingresos_b = rng.normal(38, 15, n_por_grupo)
antiguedad_b = rng.normal(6, 4, n_por_grupo)
score_b = 0.04 * ingresos_b + 0.3 * antiguedad_b - 5
etiqueta_b = (score_b + rng.normal(0, 5, n_por_grupo) > 0).astype(int)

df = pd.concat([
    pd.DataFrame({"edad": edad_a, "ingresos": ingresos_a, "antiguedad": antiguedad_a,
                  "grupo": "A", "etiqueta": etiqueta_a}),
    pd.DataFrame({"edad": edad_b, "ingresos": ingresos_b, "antiguedad": antiguedad_b,
                  "grupo": "B", "etiqueta": etiqueta_b}),
], ignore_index=True)

df.sample(5, random_state=1)

## Addestra un classificatore semplice

Adatti una regressione logistica (`LogisticRegression` di scikit-learn) sul dataset combinato, senza dirle nulla sul gruppo demografico: quella colonna non entra come variabile di input. Il modello non "sa" che esistono due gruppi; eppure, come vedrai qui sotto, può trattarli in modo diverso.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df[["edad", "ingresos", "antiguedad"]].values
y = df["etiqueta"].values
grupo = df["grupo"].values

X_train, X_test, y_train, y_test, grupo_train, grupo_test = train_test_split(
    X, y, grupo, test_size=0.3, random_state=42, stratify=grupo
)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

resultados = pd.DataFrame({"grupo": grupo_test, "y_real": y_test, "y_pred": y_pred})
resultados.head()

## Verifica l'equità del modello

Il tasso di veri positivi (TPR) misura quale proporzione dei casi positivi reali il modello identifica correttamente. La differenza di pari opportunità (equal opportunity difference) confronta il TPR tra i gruppi e sottrae il valore più basso da quello più alto.

Un valore vicino a zero indica che il modello individua con la stessa efficacia i positivi reali di entrambi i gruppi. Un valore lontano da zero segnala che uno dei due riceve sistematicamente meno veri positivi dell'altro, anche se l'algoritmo non ha mai visto quella colonna.

In [ ]:
import matplotlib.pyplot as plt


def tasa_verdaderos_positivos(subconjunto):
    """TPR = proporción de positivos reales que el modelo detecta como positivos."""
    positivos_reales = subconjunto[subconjunto["y_real"] == 1]
    if len(positivos_reales) == 0:
        return np.nan
    return (positivos_reales["y_pred"] == 1).mean()


def diferencia_oportunidad_igual(df_resultados, columna_grupo="grupo"):
    """Calcula la TPR por grupo y la diferencia entre el valor más alto y el
    más bajo: es la métrica de equal opportunity difference."""
    tprs = df_resultados.groupby(columna_grupo).apply(tasa_verdaderos_positivos)
    return tprs, tprs.max() - tprs.min()


tprs_por_grupo, diferencia = diferencia_oportunidad_igual(resultados)

print("TPR por grupo:")
print(tprs_por_grupo)
print(f"\nDiferencia de oportunidad igual: {diferencia:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
colores = ["#4C72B0", "#DD8452"]
barras = ax.bar(tprs_por_grupo.index, tprs_por_grupo.values, color=colores, width=0.5)

ax.set_ylim(0, 1)
ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
ax.set_title("Auditoría de equidad: TPR por grupo demográfico")
ax.axhline(y=tprs_por_grupo.max(), linestyle="--", color="gray", alpha=0.5)

for barra, valor in zip(barras, tprs_por_grupo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.02, f"{valor:.2f}", ha="center")

ax.text(0.5, -0.12, f"Diferencia de oportunidad igual = {diferencia:.3f}",
        transform=ax.transAxes, ha="center", fontsize=10, style="italic")

plt.tight_layout()
plt.show()

## Riepilogo dei punti chiave

- Il bias algoritmico quasi mai è intenzionale: nasce nei dati (rappresentazione, rumore di misurazione, etichette ereditate) e si manifesta nelle decisioni del modello
- Un modello può trattare un gruppo peggio di un altro anche se non riceve mai quella variabile come input: il bias filtra attraverso fattori correlati
- La differenza di pari opportunità confronta il tasso di veri positivi (TPR) tra i gruppi e ti dice se il modello individua con la stessa efficacia chi soddisfa davvero la condizione positiva
- Nessuna metrica di equità è universalmente corretta: scegli quella che corrisponde al danno concreto che vuoi evitare nel tuo caso d'uso
- Verificare un modello per gruppo demografico non è un passo facoltativo alla fine del progetto: fa parte del controllo di qualità, come l'accuratezza o la precisione

**Apri ora il quaderno di pratica di accompagnamento, `../notebooks_practica/clase12_practica_innovacion_responsable_ia.ipynb`, e metti alla prova ciò che hai appena visto.**